# 循环神经网络 LSTM

按代码单元格的原编号查阅课程手册。

In [ ]:
%matplotlib inline
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn

torch.manual_seed(42)
torch.set_num_threads(min(torch.get_num_threads(), 4))

In [ ]:
data_csv = pd.read_csv("./LSTM.csv")
data_csv.head()

In [ ]:
plt.plot(data_csv[0:200])

In [ ]:
raw_flow = data_csv['data'].dropna().to_numpy(dtype=np.float32)
step = 5
train_size = int((len(raw_flow) - step) * 0.7)
if train_size < 1 or train_size >= len(raw_flow) - step:
    raise ValueError('需要足够的数据以构造非空的训练集和测试集')
training_values = raw_flow[:train_size + step]
min_value = training_values.min()
max_value = training_values.max()
scale = max_value - min_value
if scale == 0:
    scale = np.float32(1.0)
dataset = (raw_flow - min_value) / scale

In [ ]:
def create_dataset(series, step):
    windows = np.lib.stride_tricks.sliding_window_view(series, step + 1)
    return windows[:, :-1].copy(), windows[:, -1].copy()

In [ ]:
data_X, data_Y = create_dataset(dataset, step)

In [ ]:
train_X, test_X = data_X[:train_size], data_X[train_size:]
train_Y, test_Y = data_Y[:train_size], data_Y[train_size:]

In [ ]:
train_X.shape

In [ ]:
train1 = torch.from_numpy(train_X.reshape(-1, 1, step))
train2 = torch.from_numpy(train_Y.reshape(-1, 1, 1))
test1 = torch.from_numpy(test_X.reshape(-1, 1, step))
test2 = torch.from_numpy(test_Y.reshape(-1, 1, 1))

In [ ]:
train1.shape, train2.shape, test1.shape, test2.shape

In [ ]:
# 定义模型
class lstm_linear(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=1, num_layers=2):
        super(lstm_linear, self).__init__()
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers) # rnn
        self.linear = nn.Linear(hidden_size, output_size) # 回归
        
    def forward(self, x):
        x, _ = self.lstm(x) # (seq, batch, hidden)
        s, b, h = x.shape
        x = x.view(s*b, h) # 转换成线性层的输入格式
        x = self.linear(x)
        x = x.view(s, b, -1)
        return x

In [ ]:
net = lstm_linear(step, 45)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-2)

In [ ]:
# 开始训练
for e in range(200):
    out = net(train1)
    loss = criterion(out, train2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (e + 1) % 5 == 0: 
        print('Epoch: {}, Loss: {:.5f}'.format(e + 1, loss.item()))

In [ ]:
net.eval()

In [ ]:
with torch.no_grad():
    test_predictions = net(test1)
    normalized_test_mse = criterion(test_predictions, test2).item()
print(f'Normalized test MSE: {normalized_test_mse:.6f}')

In [ ]:
pred_test = test_predictions.reshape(-1).cpu().numpy().astype(np.float64) * scale + min_value
true_test = raw_flow[train_size + step:].astype(np.float64)
last_observed = raw_flow[train_size + step - 1:-1].astype(np.float64)

In [ ]:
assert pred_test.shape == true_test.shape == last_observed.shape
print(f'Train windows: {len(train1)}, test windows: {len(test1)}')

In [ ]:
mse = np.mean((pred_test - true_test) ** 2)
rmse = np.sqrt(mse)
persistence_mse = np.mean((last_observed - true_test) ** 2)
print(f'Test MSE (people²): {mse:.6f}')
print(f'Test RMSE (people): {rmse:.6f}')
print(f'Last observation MSE (people²): {persistence_mse:.6f}')

In [ ]:
# 画出实际结果和预测的结果
plt.plot(true_test,  label='real')
plt.plot(pred_test,label='prediction')
plt.legend(loc='best')